# 02 · Advanced chunking: table, diagram, and image

Simple character-based splitting can separate a table row from its column header or lose an image. Here, we preserve the **chunk type, structure, and origin**. The Acme document contains a real Markdown table; the diagram is created locally as an explicitly labeled illustration of the method, and the vision model describes what it sees.

The model's visual description is a hypothesis to be verified. It does not replace the original file nor does it turn the illustration into a source of financial data.

In [ ]:
# The only cell with imports and project path.
import base64
import os
import sys
from pathlib import Path

import httpx
import numpy as np
from IPython.display import Image, display
from dotenv import dotenv_values
from ollama import Client
from PIL import Image as PILImage, ImageDraw, ImageFont

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import load_document, ollama_settings

## 1. A table row must know its columns

In a report, the table header explains whether `892,184.17` represents the number of invoices, an amount, or a date. A single row as standalone text is illegible. We copy the headers to every record and preserve the document ID, version, and section. The `---` separator in Markdown is not data.

In [ ]:
doc = load_document("FIN-YTD-2011")
table = doc["body"].split("## Monthly evidence table", 1)[1].split("## ", 1)[0]
lines = [line for line in table.splitlines() if line.startswith("|")]
columns = [cell.strip() for cell in lines[0].strip("|").split("|")]
rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in lines[2:]]
table_chunks = [
    {"id": f"FIN-YTD-2011:v1:table:r{i:02d}", "modality": "table_row", "source": doc["path"].relative_to(ROOT).as_posix(),
     "text": "Monthly evidence table. " + "; ".join(f"{key}: {value}" for key, value in zip(columns, row))}
    for i, row in enumerate(rows)
]
table_chunks[2]

The answer regarding the November billing can now fit in a single line, but explaining the difference from October requires **two** lines and a paragraph of interpretation. Tables are often indexed in two ways: row records for exact matches and a description of the entire table for broader queries.

In [ ]:
table_summary = {"id": "FIN-YTD-2011:v1:table:summary", "modality": "table_summary",
                 "source": doc["path"].relative_to(ROOT).as_posix(),
                 "text": "Monthly evidence table: invoice count and header billing GBP for September, October, November and partial December 2011. December is incomplete."}
[table_summary, table_chunks[1], table_chunks[2]]

## 2. Diagram: pixels + caption + relation to source

We create an image showing the stages of the backtest. This is an **illustration prepared for lesson purposes** based on the note `SCI-FORECAST-2011:v1`, not a new research result. In the RAG system, we store the image path, caption, origin, and version. When a question requires numbers or methodology, we refer back to the source text.

In [ ]:
image_path = ROOT / ".tmp" / "rag" / "forecast_flow.png"
image_path.parent.mkdir(parents=True, exist_ok=True)
canvas = PILImage.new("RGB", (960, 230), "white")
draw = ImageDraw.Draw(canvas)
font = ImageFont.load_default(size=16)
labels = ["Freeze cutoff", "Train baseline", "Forecast next period", "Compare actuals"]
for i, label in enumerate(labels):
    left = 18 + i * 238
    draw.rounded_rectangle((left, 65, left + 205, 155), radius=14, outline="#2456A6", width=3, fill="#EAF2FF")
    draw.text((left + 12, 101), label, fill="#15305A", font=font)
    if i < 3:
        draw.line((left + 207, 110, left + 233, 110), fill="#2456A6", width=4)
        draw.polygon([(left + 233, 110), (left + 222, 103), (left + 222, 117)], fill="#2456A6")
canvas.save(image_path)
display(Image(filename=str(image_path)))

## 3. Vision model reads the image

`gemma4:12b` has local vision and chat functions; the name can be changed via `RAG_VISION_MODEL` in the **root `.env`**. `clef` and `clef-flash` are **System One decision-making** models, so we do not use their chat endpoint for signatures. We verify the result with the labels we drew ourselves. If the model skips or swaps a stage, correct the signature before indexing. In a real PDF, similarly preserve the page number, coordinates, and adjacent signature.

In [ ]:
host, _ = ollama_settings()
vision_model = os.environ.get("RAG_VISION_MODEL") or dotenv_values(ROOT / ".env").get("RAG_VISION_MODEL") or "gemma4:12b"
vision_caption = Client(host=host).chat(
    model=vision_model,
    messages=[{"role": "user", "content": "Transcribe the four box labels from left to right, then describe the arrows in one sentence. Do not infer financial results.",
               "images": [str(image_path)]}],
    options={"temperature": 0}, think=False,
).message.content.strip()
missing_labels = [label for label in labels if label.lower() not in vision_caption.lower()]
{"model_caption": vision_caption, "missing_labels": missing_labels}

`missing_labels` shows whether the model has read all stages. Since we created the diagram ourselves, we build the description for the index using **explicit labels and arrows from the code**, while keeping the raw model caption for comparison. For an external image, we do not have such a source of labels: human oversight or OCR verified against the image is required.

In [ ]:
# This diagram was created from explicit labels and arrows: this description can be verified without a model.
verified_caption = "Four boxes from left to right: " + " -> ".join(labels) + ". Arrows connect adjacent boxes."
diagram_chunk = {"id": "SCI-FORECAST-2011:v1:diagram:lesson", "modality": "diagram_caption",
                 "source": "SCI-FORECAST-2011:v1 (illustration, not source evidence)",
                 "image_path": image_path.relative_to(ROOT).as_posix(), "text": verified_caption,
                 "model_caption": vision_caption, "model_omitted_labels": missing_labels}
{"types": [chunk["modality"] for chunk in [table_summary, table_chunks[0], diagram_chunk]],
 "indexed_diagram_caption": diagram_chunk["text"], "model_omitted_labels": missing_labels}

## 4. True image and text embeddings in a single space

`embeddinggemma-2:latest` accepts image and text via `/api/embed` and returns corresponding 768D vectors. In this version, the Ollama Python library types `Client.embed` as text input, so for multimodal requests, we explicitly use the official HTTP API. We compare the **diagram pixel** vector with two texts. This is a different model and a different space than `qwen3-embedding:0.6b`; never mix their vectors in the same index.

In [ ]:
multimodal_model = os.environ.get("RAG_MULTIMODAL_EMBED_MODEL") or dotenv_values(ROOT / ".env").get("RAG_MULTIMODAL_EMBED_MODEL") or "embeddinggemma-2:latest"
image_base64 = base64.b64encode(image_path.read_bytes()).decode("ascii")
payload = {"model": multimodal_model, "input": [
    {"image": image_base64},
    {"text": "A four-step diagram: freeze cutoff, train baseline, forecast next period, compare actuals"},
    {"text": "A ledger of customer payment receipts and invoice allocations"},
], "truncate": False}
response = httpx.post(host.rstrip("/") + "/api/embed", json=payload, timeout=180)
response.raise_for_status()
mixed_vectors = np.asarray(response.json()["embeddings"], dtype=np.float32)
mixed_unit = mixed_vectors / np.linalg.norm(mixed_vectors, axis=1, keepdims=True)
{"shape": mixed_vectors.shape, "diagram_vs_matching_text": float(mixed_unit[0] @ mixed_unit[1]),
 "diagram_vs_unrelated_text": float(mixed_unit[0] @ mixed_unit[2])}

## 5. Multimodal Boundary

We have two paths: (1) image → verified caption → text embedding; (2) pixels and text → **shared space** of a multimodal model. A raw model caption may lose the layout, arrows, or small text. An image vector is also not proof that the model read every label. Store the original and show it to the user.

**Exercise:** draw a feedback loop between the stages, repeat the caption, and check if the model notices it. For the table, remove the header from the row text and compare the interpretation of the number `892,184.17`.